# Прогнозування доходу українських підприємств

Єдиний ноутбук проєкту: підготовка даних, валідація, базові моделі, EDA, Ridge M2, LightGBM M3, аналіз сегментів і висновки.

# Частина A. Дані, панель, валідація та базові моделі

# 0. Дані та ціль

**Джерело:** data.gov.ua, набір даних Держстату «Фінансова звітність підприємств (річна)». Для 2021–2024 років повний базовий зріз об’єднується з поточним пакетом пізніх подань; після дедуплікації залишається найновіше подання підприємства. Файл за 2025 рік не читається.

**Ціль:** z = log1p(чистий дохід у році t+1) у цінах 2021 року. Популяція: підприємства з доходом y_t > 0, які подали звіт за t+1.

## Налаштування

In [64]:
!pip -q install -r requirements.txt
import io, os, re, subprocess, time, zipfile
from pathlib import Path
from getpass import getpass

import numpy as np
import polars as pl
import requests
import statsmodels.api as sm
from huggingface_hub import HfApi, hf_hub_download, login

ROOT = Path('/content') if Path('/content').exists() else Path('.')
RAW, OUT = str(ROOT / 'raw'), str(ROOT / 'out')
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

HF_REPO = 'Shnnapka/fin-zvit-ua'
REBUILD = True    # повний відтворюваний запуск для актуальної кумулятивної вибірки
UPLOAD = False    # True: завантажити звіти й панель у HF_REPO (запитає токен)

BASE = 'https://data.gov.ua/dataset/6456ef0c-985c-4bb7-bf9d-16831161b416/resource'
API = 'https://data.gov.ua/api/3/action/resource_show'

# Кожен рік складається з повного базового зрізу та поточного пакета пізніх
# подань. Поточні ресурси data.gov.ua не є кумулятивними, тому використовувати
# їх окремо не можна. latest() нижче залишає найновіше подання підприємства.
SOURCES = {
    2021: [
        ('rep2021_base', '2f24df98-79ea-443f-9b89-e8cd3f1b9ae2', 397299),
        ('rep2021_late_current', '2f24df98-79ea-443f-9b89-e8cd3f1b9ae2', None),
    ],
    2022: [
        ('rep2022_f1_base', '1e93dd50-f137-43f1-a165-6d7fd8f2c329', None),
        ('rep2022_f2_base', '4cee01d7-a483-4b28-88fb-4dfe770afe2c', None),
        ('rep2022_m_base',  '70d2d63c-a3c0-4818-bb7f-f6ab5854c7b0', None),
        ('rep2022_ms_base', '4d214aeb-b9ae-4d80-a220-3b585bf1e90c', None),
        ('rep2022_late_current', 'b2e27b15-c3cc-4944-9d39-86260665c9a9', None),
    ],
    2023: [
        ('rep2023_base', '9e7b6a7d-eaeb-44e2-9947-e34b25ce83cf', 397278),
        ('rep2023_late_current', '9e7b6a7d-eaeb-44e2-9947-e34b25ce83cf', None),
    ],
    2024: [
        ('rep2024_base', 'c58bf087-77b1-4355-9879-16d8cea35677', 442489),
        ('rep2024_late_current', 'c58bf087-77b1-4355-9879-16d8cea35677', None),
    ],
}

# Звіт про фінансові результати малих і мікро: _3 = звітний рік, _4 = попередній (див. «Перевірка відповідності колонок»).
SMALL_INCOME_CUR, SMALL_INCOME_PREV = '_3', '_4'

def load_parquet(name, hf_dir):
    """Читає OUT/name; якщо файлу немає, завантажує HF_REPO/hf_dir/name і зберігає локальну копію."""
    local = f'{OUT}/{name}'
    if not os.path.exists(local):
        path = hf_hub_download(HF_REPO, f'{hf_dir}/{name}', repo_type='dataset')
        pl.read_parquet(path).write_parquet(local)
    return pl.read_parquet(local)

_hf_api = None

def hf_upload(pattern, path_in_repo):
    """Завантажує файли з OUT, що відповідають `pattern`, у HF_REPO/path_in_repo. Нічого не робить, якщо UPLOAD не True."""
    global _hf_api
    if not UPLOAD:
        print(f'UPLOAD = False: skipped {path_in_repo}/{pattern}')
        return
    if _hf_api is None:
        login(token=getpass('Paste your HF token and press Enter: '))
        _hf_api = HfApi()
        _hf_api.create_repo(HF_REPO, repo_type='dataset', private=False, exist_ok=True)
    _hf_api.upload_folder(folder_path=OUT, repo_id=HF_REPO, repo_type='dataset',
                          path_in_repo=path_in_repo, allow_patterns=pattern)
    print('✓ Uploaded:', f'https://huggingface.co/datasets/{HF_REPO}/tree/main/{path_in_repo}')

## Функція завантаження

In [65]:
def download(name, rid, rev):
    dst = f'{RAW}/{name}.zip'
    if os.path.exists(dst + '.ok'):
        return dst
    if rev is None:
        url = requests.get(API, params={'id': rid}, timeout=60).json()['result']['url']
    else:
        url = f'{BASE}/{rid}/revision/{rev}/download'
    print(f'   ↓ {name} ...', flush=True)
    subprocess.run(['wget', '-q', '-c', '-O', dst, url], check=True)
    zipfile.ZipFile(dst).close()            # помилка, якщо архів обрізаний
    open(dst + '.ok', 'w').close()
    print(f'   ✓ {name}: {os.path.getsize(dst) / 1e6:.0f} MB', flush=True)
    return dst

## Розбір XML в один рядок на підприємство й рік

Використані коди рядків (національні стандарти бухгалтерського обліку, НП(С)БО 1 і 25):

| Код | Поле | Форма |
|---|---|---|
| 1300 | `assets`: усього активів | баланс |
| 1495 | `equity`: усього власного капіталу | баланс |
| 1595 | `lt_liab`: довгострокові зобов'язання | баланс |
| 1695 | `cur_liab`: поточні зобов'язання | баланс |
| 2000 | `revenue`: чистий дохід (**ціль**) | звіт про фінансові результати |
| 2350 / 2355 | `net_result`: чистий прибуток / збиток (повна форма: 2350 − 2355) | звіт про фінансові результати |
| `S_CH` | `employees` (лише малі та мікропідприємства) | заголовок |

Відповідність колонок (перевірено нижче): повний баланс `B` = кінець року, `A` = початок; повний звіт про фінансові результати `A` = поточний рік, `B` = попередній;
баланс малих і мікропідприємств `_4` = кінець, `_3` = початок; їхній звіт про фінансові результати `_3` = поточний рік, `_4` = попередній.

In [66]:
FORM_CODES = [('S01001', 'f1'), ('S01002', 'f2'), ('S01100', 'm'), ('S01110', 'ms')]
TAG = re.compile(rb'<([A-Z0-9_]+)>([^<]*)</\1>')

def form_of(name):
    for code, form in FORM_CODES:
        if code in name:
            return form
    return None

def iter_xml(path):
    """Повертає (форма, байти XML) з архіву, зокрема із вкладених zip."""
    with zipfile.ZipFile(path) as outer:
        for ii in outer.infolist():
            n = ii.filename
            if n.lower().endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(outer.read(ii))) as inner:
                    for jj in inner.infolist():
                        f = form_of(jj.filename)
                        if f and jj.filename.lower().endswith('.xml'):
                            yield f, inner.read(jj)
            else:
                f = form_of(n)
                if f and n.lower().endswith('.xml'):
                    yield f, outer.read(ii)

def parse(b):
    """Плаский словник {тег: текст}; порожній, якщо у файлі немає XML."""
    # Файли 2022 року загортають XML у цифровий підпис, тому вирізаємо сам XML.
    s, e = b.find(b'<?xml'), b.rfind(b'</DECLAR>')
    if s < 0:
        return {}
    return {k.decode(): v.decode('utf-8', 'replace') for k, v in TAG.findall(b[s:e])}

In [67]:
def num(d, k):
    v = d.get(k)
    if v is None or v.strip() == '':
        return None
    try:
        return float(v.replace(',', '.'))
    except ValueError:
        return None

def diff(a, b):
    return None if a is None and b is None else (a or 0.0) - (b or 0.0)

def header(d):
    edr = (d.get('FIRM_EDRPOU') or d.get('TIN') or '').strip()
    fill = d.get('D_FILL', '')                      # DDMMYYYY -> YYYYMMDD
    fill = int(fill[4:8] + fill[2:4] + fill[0:2]) if len(fill) == 8 and fill.isdigit() else 0
    cnt = d.get('C_DOC_CNT', '')
    return {
        'edrpou': edr.zfill(8) if edr.isdigit() else edr,
        'year': int(d['PERIOD_YEAR']) if d.get('PERIOD_YEAR', '').isdigit() else None,
        'period_type': d.get('PERIOD_TYPE'),
        'd_fill': fill,
        'doc_cnt': int(cnt) if cnt.isdigit() else 0,
        'kved': (d.get('FIRM_KVED') or d.get('KVED') or '').strip() or None,
        'obl': (d.get('OBL') or d.get('C_REG') or '').strip() or None,
    }

def row(form, d):
    r = header(d)
    if form == 'f1':     # повний баланс: B = кінець року, A = початок
        r.update(assets=num(d, 'B1300'), assets_prev=num(d, 'A1300'),
                 equity=num(d, 'B1495'), equity_prev=num(d, 'A1495'),
                 lt_liab=num(d, 'B1595'), lt_liab_prev=num(d, 'A1595'),
                 cur_liab=num(d, 'B1695'), cur_liab_prev=num(d, 'A1695'),
                 n3=num(d, 'N3'))
    elif form == 'f2':   # повний звіт про фін. результати: A = поточний рік, B = попередній
        r.update(revenue=num(d, 'A2000'), revenue_prev=num(d, 'B2000'),
                 net_result=diff(num(d, 'A2350'), num(d, 'A2355')),
                 net_result_prev=diff(num(d, 'B2350'), num(d, 'B2355')))
    else:                # малі й мікро: баланс _4 = кінець, _3 = початок
        c, p = SMALL_INCOME_CUR, SMALL_INCOME_PREV
        r.update(assets=num(d, 'A1300_4'), assets_prev=num(d, 'A1300_3'),
                 equity=num(d, 'A1495_4'), equity_prev=num(d, 'A1495_3'),
                 lt_liab=num(d, 'A1595_4'), lt_liab_prev=num(d, 'A1595_3'),
                 cur_liab=num(d, 'A1695_4'), cur_liab_prev=num(d, 'A1695_3'),
                 revenue=num(d, 'B2000' + c), revenue_prev=num(d, 'B2000' + p),
                 net_result=num(d, 'B2350' + c), net_result_prev=num(d, 'B2350' + p),
                 employees=num(d, 'S_CH'))
    return r

def latest(df):
    """Один звіт на (ЄДРПОУ, рік): найпізніше подання."""
    return (df.sort(['edrpou', 'year', 'd_fill', 'doc_cnt'])
              .unique(subset=['edrpou', 'year'], keep='last', maintain_order=True))

## Побудова однієї таблиці на звітний рік

Повна форма надходить двома файлами (баланс f1, звіт про фінансові результати f2), які об'єднуються за підприємством.
Якщо підприємство подало кілька форм за один рік, береться найпізніше подання.

In [68]:
def build_year(year):
    t0 = time.time()
    rows = {'f1': [], 'f2': [], 'm': [], 'ms': []}
    n = skipped = 0
    for name, rid, rev in SOURCES[year]:
        path = download(name, rid, rev)
        for form, b in iter_xml(path):
            d = parse(b)
            if not d:
                skipped += 1
                continue
            if d.get('PERIOD_TYPE') != '5':        # 5 = річний звіт
                continue
            rows[form].append(row(form, d))
            n += 1
            if n % 100_000 == 0:
                print(f'      {year}: read {n:,} reports ({time.time() - t0:.0f} s)', flush=True)

    dfs = {k: latest(pl.DataFrame(v, infer_schema_length=None)) for k, v in rows.items() if v}
    other = {k: df.filter(pl.col('year') != year).height for k, df in dfs.items()}
    dfs = {k: df.filter(pl.col('year') == year) for k, df in dfs.items()}
    for k in ['f1', 'f2', 'm', 'ms']:
        if k not in dfs:
            print(f'   ⚠ {year}: no reports of form {k}', flush=True)

    parts = []
    if 'f1' in dfs and 'f2' in dfs:
        full = (dfs['f1'].join(dfs['f2'].select(['edrpou', 'year', 'revenue', 'revenue_prev',
                                                 'net_result', 'net_result_prev', 'd_fill', 'kved']),
                               on=['edrpou', 'year'], how='full', coalesce=True, suffix='_f2')
                .with_columns(pl.max_horizontal('d_fill', 'd_fill_f2').alias('d_fill'),
                              pl.coalesce('kved', 'kved_f2').alias('kved'))
                .drop(['d_fill_f2', 'kved_f2']))
    else:
        full = dfs.get('f1', dfs.get('f2'))
    if full is not None:
        parts.append(full.with_columns(pl.lit('full').alias('form')))
    if 'm' in dfs:
        parts.append(dfs['m'].with_columns(pl.lit('small').alias('form')))
    if 'ms' in dfs:
        parts.append(dfs['ms'].with_columns(pl.lit('micro').alias('form')))

    allf = pl.concat(parts, how='diagonal_relaxed')
    sizes = {f: allf.filter(pl.col('form') == f).height for f in ['full', 'small', 'micro']}
    before = allf.height
    allf = latest(allf)
    allf.write_parquet(f'{OUT}/reports_{year}.parquet')
    print(f'✓ {year}: {allf.height:,} companies (full {sizes["full"]:,}, small {sizes["small"]:,}, '
          f'micro {sizes["micro"]:,}; cross-form duplicates {before - allf.height:,}; '
          f'other-year reports dropped {other}; files without XML {skipped:,}) — {time.time() - t0:.0f} s',
          flush=True)
    return allf

## Завантаження та розбір звітів за 2021–2024 роки

In [ ]:
if REBUILD:
    years = {y: build_year(y) for y in [2021, 2022, 2023, 2024]}
    pl.concat(list(years.values()), how='diagonal_relaxed').write_parquet(f'{OUT}/reports_all.parquet')
    for y in years:
        print(f'   reports_{y}.parquet: {os.path.getsize(f"{OUT}/reports_{y}.parquet") / 1e6:.1f} MB')
    rep = pl.read_parquet(f'{OUT}/reports_all.parquet')
else:
    print('REBUILD = False: skipped download and parsing, using reports_all.parquet')
    rep = load_parquet('reports_all.parquet', 'reports')
    years = {y: rep.filter(pl.col('year') == y) for y in [2021, 2022, 2023, 2024]}

print(rep.group_by('year', 'form').len().pivot(on='form', index='year', values='len')
         .with_columns(pl.sum_horizontal('full', 'small', 'micro').alias('total'))
         .select(['year', 'full', 'small', 'micro', 'total']).sort('year'))

**Висновок.** Повний річний зріз об’єднується з поточним пакетом пізніх подань, після чого для кожного підприємства залишається найновіший звіт. Такий підхід зберігає повноту базового файлу й одночасно враховує доступні на дату запуску виправлення. Окремий файл 2021 року дає фактичні значення за 2021 рік; колонки «попередній рік» більше не підміняють річний файл.

## Перевірка відповідності колонок

Графа «попередній рік» у звіті за рік Y має збігатися з графою «поточний рік» у звіті того самого підприємства за рік Y−1.

In [ ]:
def check(cur, prev):
    j = cur.join(prev, on='edrpou', suffix='_y1')
    for form in ['full', 'small', 'micro']:
        s = j.filter(pl.col('form') == form)
        for a, b in [('revenue_prev', 'revenue_y1'), ('revenue', 'revenue_y1'),
                     ('assets_prev', 'assets_y1')]:
            ok = s.filter(pl.col(a).is_not_null() & pl.col(b).is_not_null())
            if ok.height:
                share = ok.filter((pl.col(a) - pl.col(b)).abs() < 0.6).height / ok.height  # значення в тис. грн, округлені
                print(f'   {form:6s} {a:13s}(Y) = {b:11s}(Y-1)?  {share:6.1%}  ({ok.height:,} companies)')

print('2024 vs 2023:'); check(years[2024], years[2023])
print('\n2023 vs 2022:'); check(years[2023], years[2022])

**Висновок.** Графа «попередній рік» у звіті за рік `t` переважно збігається з окремим звітом за `t−1`, тоді як графа поточного року — ні. Отже, мапінг колонок правильний. Основні значення 2021 року беруться з окремого річного файлу, а графа «попередній рік» використовується лише як доступний у році `t` лаг.

## Завантаження звітів на Hugging Face

In [ ]:
hf_upload('reports_*.parquet', 'reports')

# 1. Підготовка даних

## Дефляція до цін 2021 року

Середньорічний індекс споживчих цін (рік до попереднього року), Держстат. Значення за 2024 рік ще треба звірити на ukrstat.gov.ua.

In [ ]:
CPI_YOY = {2022: 1.202, 2023: 1.129, 2024: 1.065}
DEFL = {2021: 1.0}
for y in [2022, 2023, 2024]:
    DEFL[y] = DEFL[y - 1] * CPI_YOY[y]
print('Deflator (2021 prices = 1):', {k: round(v, 4) for k, v in DEFL.items()})

## Від розділу КВЕД до секції

In [73]:
# Діапазони розділів КВЕД-2010 для кожної літери секції (як у NACE Rev. 2).
SECTIONS = [('A', 1, 3), ('B', 5, 9), ('C', 10, 33), ('D', 35, 35), ('E', 36, 39), ('F', 41, 43),
            ('G', 45, 47), ('H', 49, 53), ('I', 55, 56), ('J', 58, 63), ('K', 64, 66), ('L', 68, 68),
            ('M', 69, 75), ('N', 77, 82), ('O', 84, 84), ('P', 85, 85), ('Q', 86, 88), ('R', 90, 93),
            ('S', 94, 96), ('T', 97, 98), ('U', 99, 99)]

def kved_section(col='kved'):
    div = pl.col(col).str.extract(r'^(\d{1,2})', 1).cast(pl.Int32, strict=False)
    expr = pl.lit(None, dtype=pl.Utf8)
    for s, lo, hi in reversed(SECTIONS):
        expr = pl.when(div.is_between(lo, hi)).then(pl.lit(s)).otherwise(expr)
    return expr

## Значення за підприємством і роком та лаг доходу

In [ ]:
rep = load_parquet('reports_all.parquet', 'reports')
assert rep['year'].max() <= 2024, 'Data contains 2025 (holdout)!'

VARS = ['revenue', 'assets', 'equity', 'lt_liab', 'cur_liab', 'net_result']

own = (rep.select(['edrpou', 'year', 'form', 'kved', 'obl', 'employees'] + VARS)
          .with_columns(pl.lit('own').alias('src')))

assert 2021 in rep['year'].unique().to_list(), 'Rebuild reports_all.parquet with the explicit 2021 source'
vals = own

deflator = pl.col('year').replace_strict(DEFL, return_dtype=pl.Float64)
vals = vals.with_columns([(pl.col(v) / deflator).alias(v) for v in VARS])

lag = (rep.select(['edrpou', 'year', 'revenue_prev'])
          .with_columns((pl.col('revenue_prev') /
                         (pl.col('year') - 1).replace_strict(DEFL, default=None, return_dtype=pl.Float64))
                        .alias('revenue_lag'))
          .select(['edrpou', 'year', 'revenue_lag']))
print(vals.group_by('year', 'src').len().sort('year'))

## Побудова цілі та панелі

Один рядок на підприємство й рік t, ціль береться зі звіту t+1.
Порожній рядок доходу в поданому звіті про фінансові результати вважається нулем. Звіт повної форми з балансом, але без звіту про фінансові результати відкидається, бо його дохід невідомий, а не нульовий.
Від'ємний лаг замінюється на null.

In [ ]:
INCOME = ['revenue', 'revenue_prev', 'net_result', 'net_result_prev']
no_income = (rep.filter((pl.col('form') == 'full') & pl.all_horizontal([pl.col(c).is_null() for c in INCOME]))
                .select(['edrpou', (pl.col('year') - 1).alias('year')]))

target = (vals.filter(pl.col('src') == 'own')
              .select(['edrpou', (pl.col('year') - 1).alias('year'),
                       pl.col('revenue').fill_null(0).alias('y_next'), pl.col('form').alias('form_next')])
              .filter(pl.col('year') >= 2021)
              .join(no_income, on=['edrpou', 'year'], how='anti'))

panel = (vals.filter(pl.col('year').is_between(2021, 2023))
             .join(lag, on=['edrpou', 'year'], how='left')
             .join(target, on=['edrpou', 'year'], how='inner')
             .filter(pl.col('revenue') > 0)
             .rename({'year': 't', 'revenue': 'y_t'})
             .with_columns(
                 (pl.col('t') + 1).alias('target_year'),
                 kved_section('kved').alias('kved_section'),
                 pl.col('y_t').log1p().alias('z_t'),
                 pl.when(pl.col('revenue_lag') >= 0).then(pl.col('revenue_lag').log1p()).alias('z_lag'),
                 pl.col('y_next').clip(lower_bound=0).log1p().alias('z_next'),
             )
             .with_columns((pl.col('z_next') - pl.col('z_t')).alias('dz_next'))
             .sort(['edrpou', 't']))

assert panel['target_year'].max() <= 2024, 'Target for 2025 is the holdout!'
assert (panel['t'] < panel['target_year']).all()

print(f'Full-form reports without an income statement (not used as targets): {no_income.height:,}')
print(panel.group_by('target_year').agg(pl.len().alias('rows'),
                                        pl.col('z_lag').is_null().mean().alias('share_no_lag'),
                                        pl.col('kved_section').is_null().mean().alias('share_no_kved'),
                                        pl.col('dz_next').median().alias('median_dz')).sort('target_year'))

**Висновок.** Панель будується лише з окремих річних файлів 2021–2024 років. Лаг за 2020 рік береться з дозволеної графи «попередній рік» у звіті за 2021 рік; його пропуски тепер є властивістю конкретних звітів, а не наслідком відсутнього річного файлу. Ціль завжди походить із наступного року, а ознаки — не пізніше року `t`.

## Збереження та завантаження панелі

In [ ]:
panel.write_parquet(f'{OUT}/panel_v1.parquet')

sample = panel.filter(pl.col('edrpou').hash(seed=42) % 1000 < 6)
sample.write_parquet(f'{OUT}/panel_v1_sample.parquet')

print(f"panel_v1: {panel.height:,} rows, {panel['edrpou'].n_unique():,} companies, "
      f"{os.path.getsize(f'{OUT}/panel_v1.parquet') / 1e6:.1f} MB")
print(f"sample:   {sample.height:,} rows, {sample['edrpou'].n_unique():,} companies")
print(panel.columns)

hf_upload('panel_v1*.parquet', 'panel')

**Висновок.** Панель зберігається як окремий артефакт і використовується всіма моделями без повторної зміни популяції. Кількість рядків і підприємств друкується під час запуску, тому висновок не фіксує значення, які можуть змінитися після додавання пізніх подань.

# 2. Схема валідації

Розширюване вікно, горизонт один рік: **F1** навчається на цілі 2022 і тестується на 2023; **F2** навчається на цілях 2022–2023 і тестується на 2024.

## Завантаження панелі та визначення фолдів

In [ ]:
panel = load_parquet('panel_v1.parquet', 'panel')

FOLDS = {'F1': ([2022], 2023), 'F2': ([2022, 2023], 2024)}

def split(fold):
    train_years, test_year = FOLDS[fold]
    tr = panel.filter(pl.col('target_year').is_in(train_years))
    te = panel.filter(pl.col('target_year') == test_year)
    assert tr['target_year'].max() < test_year     # цілі навчання строго раніше за тестовий рік
    assert te['t'].max() < test_year                # тестові ознаки походять із років ≤ t
    return tr, te

## Підприємства за цільовим роком і формою

In [ ]:
print(panel.group_by('target_year', 'form').len().pivot(on='form', index='target_year', values='len')
           .sort('target_year'))

**Висновок.** Основну частину панелі формують малі та мікропідприємства, а повна форма є значно меншою групою. Тому загальна метрика насамперед характеризує малі форми, а якість для повної форми перевіряється окремо в сегментному аналізі. Порівняння F1 і F2 виконується на однаково визначеній популяції, але можливий часовий зсув структури враховується як обмеження.

# 3. Бенчмарки та метрики

## Метрики

`MAE_log` — основна метрика (середня абсолютна помилка z). `Bias` = mean(ẑ − z): від'ємне значення означає недооцінку.
`Skill` = 1 − MAE / MAE(N1), тобто додатне значення означає кращий результат, ніж у персистентності. `MdAPE` і `WAPE` рахуються у шкалі доходу; MdAPE використовує лише y > 0.

In [ ]:
def metrics(z, z_hat, z_n1=None):
    z, z_hat = np.asarray(z, float), np.asarray(z_hat, float)
    y, y_hat = np.expm1(z), np.expm1(z_hat)
    mae = np.mean(np.abs(z - z_hat))
    pos = y > 0
    out = {
        'MAE_log': mae,
        'Bias': np.mean(z_hat - z),
        'MdAPE': np.median(np.abs(y[pos] - y_hat[pos]) / y[pos]),
        'WAPE': np.sum(np.abs(y - y_hat)) / np.sum(y),
        'n': len(z),
    }
    if z_n1 is not None:
        out['Skill'] = 1 - mae / np.mean(np.abs(z - np.asarray(z_n1, float)))
    return out

## Бенчмарки N1–N3

- **N1**, персистентність: ẑ = z_t.
- **N2**, галузевий дрейф: z_t плюс медіанний ріст z_t − z_lag секції КВЕД підприємства в році t (медіана по країні, якщо секції немає).
- **N3**, середнє двох років: (z_t + z_lag) / 2 або z_t, якщо лага немає.

Медіани N2 використовують лише дані року t, відомі в момент t.

In [ ]:
def sector_growth(df):
    g = df.filter(pl.col('z_lag').is_not_null()).with_columns((pl.col('z_t') - pl.col('z_lag')).alias('dz_t'))
    by_sec = g.group_by('t', 'kved_section').agg(pl.col('dz_t').median().alias('g_sec'))
    by_all = g.group_by('t').agg(pl.col('dz_t').median().alias('g_all'))
    return by_sec, by_all

def predict_naive(te):
    by_sec, by_all = sector_growth(te)
    te = te.join(by_sec, on=['t', 'kved_section'], how='left').join(by_all, on='t', how='left')
    return te.with_columns(
        pl.col('z_t').alias('N1'),
        (pl.col('z_t') + pl.coalesce('g_sec', 'g_all', pl.lit(0.0))).alias('N2'),
        pl.when(pl.col('z_lag').is_not_null()).then((pl.col('z_t') + pl.col('z_lag')) / 2)
          .otherwise(pl.col('z_t')).alias('N3'),
    )

## Модель M1: панельна AR(1)

z_next = α + β·z_t, спільна для всіх підприємств. Основна модель оцінюється методом найменших квадратів (OLS) на всіх рядках навчання.

**Додатковий аналіз: M1-LAD.** Та сама специфікація, оцінена медіанною регресією (абсолютна втрата, що відповідає `MAE_log`). Вона показує вплив функції втрат і не є окремою моделлю порівняння, хоча для зручності стоїть у таблиці нижче.

In [ ]:
def fit_m1(tr):
    X = sm.add_constant(tr['z_t'].to_numpy())
    return sm.OLS(tr['z_next'].to_numpy(), X).fit()

def fit_m1_lad(tr):
    X = sm.add_constant(tr['z_t'].to_numpy())
    return sm.QuantReg(tr['z_next'].to_numpy(), X).fit(q=0.5)

def predict_m1(model, te):
    return model.predict(sm.add_constant(te['z_t'].to_numpy(), has_constant='add'))

## Оцінювання на F1 і F2

In [ ]:
MODELS = ['N1', 'N2', 'N3', 'M1', 'M1-LAD']
rows, preds = [], {}
for fold in FOLDS:
    tr, te = split(fold)
    te = predict_naive(te)
    ols, lad = fit_m1(tr), fit_m1_lad(tr)
    te = te.with_columns(pl.Series('M1', predict_m1(ols, te)), pl.Series('M1-LAD', predict_m1(lad, te)))
    print(f'{fold}: train {tr.height:,}, test {te.height:,}')
    print(f'    M1      z_next = {ols.params[0]:.3f} + {ols.params[1]:.3f}·z_t   (R² = {ols.rsquared:.3f})')
    print(f'    M1-LAD  z_next = {lad.params[0]:.3f} + {lad.params[1]:.3f}·z_t')
    for name in MODELS:
        rows.append({'model': name, 'fold': fold, **metrics(te['z_next'], te[name], te['N1'])})
    preds[fold] = te

results = pl.DataFrame(rows)
summary = (results.group_by('model')
                  .agg(pl.col('MAE_log').mean().alias('MAE_log (mean F1–F2)'),
                       pl.col('Skill').min().alias('Skill (min)'))
                  .sort('MAE_log (mean F1–F2)'))
pl.Config.set_tbl_rows(30); pl.Config.set_float_precision(4)
print(results.select(['model', 'fold', 'MAE_log', 'Skill', 'Bias', 'MdAPE', 'WAPE', 'n']).sort('fold', 'model'))
print(summary)
results.write_parquet(f'{OUT}/results_A.parquet')

**Висновок.** На контрольному опублікованому зрізі N1 має найменший `MAE_log` серед N1–N3 і M1 в обох часових фолдах. M1-OLS систематично занижує прогноз: на її середню квадратичну функцію втрат впливають великі від’ємні зміни підприємств, що обнулили дохід, а залежність, оцінена на одному часовому режимі, нестабільно переноситься на наступний. M1-LAD зменшує помилку приблизно на 18%, що підтверджує вплив хвостів розподілу, але не перевершує N1. Остаточний вибір робиться лише після спільного порівняння N1–N3 і M1–M3.

# 4. Перевірки на витоки

## Перевірки панелі та фолдів

Кожна перевірка є `assert`, тому Run All зупиниться, якщо будь-яка з них не пройде.

In [ ]:
def check_panel(panel):
    # 1. Ознаки лише з років <= t, ціль — t+1
    assert (panel['t'] < panel['target_year']).all(), 'feature year >= target year'
    assert (panel['target_year'] - panel['t'] == 1).all(), 'horizon is not 1 year'
    # 2. Holdout 2025 ніколи не використовується
    assert panel['target_year'].max() <= 2024 and panel['t'].max() <= 2023, '2025 found'
    # 3. Один рядок на підприємство й рік
    assert panel.select(['edrpou', 't']).is_unique().all(), 'duplicate company-year rows'
    # 4. Графи попереднього року з файлу t+1 ніколи не є ознаками:
    #    лише t = 2021 береться з граф попереднього року (звітів за 2022 рік)
    bad = panel.filter((pl.col('t') >= 2022) & (pl.col('src') != 'own')).height
    assert bad == 0, f'{bad} rows with t >= 2022 use previous-year columns'
    # 5. Немає фільтра «присутні в усіх роках»: панель має бути незбалансованою
    share_all = (panel.group_by('edrpou').len()['len'] == panel['t'].n_unique()).mean()
    assert share_all < 1, 'every firm appears in every year -> survivor filter?'
    print(f'Panel checks passed. Firms present in all {panel["t"].n_unique()} years: {share_all:.1%} '
          f'(the rest enter or exit, so there is no survivor filter).')

def check_folds(panel, folds, split):
    for fold, (train_years, test_year) in folds.items():
        tr, te = split(fold)
        assert tr['target_year'].max() < test_year, f'{fold}: train overlaps test in time'
        assert te['t'].max() < test_year, f'{fold}: test features from the test year'
        assert te.filter(pl.col('src') != 'own').height == 0, f'{fold}: test uses previous-year columns'
        print(f'{fold}: train targets {sorted(tr["target_year"].unique().to_list())}, '
              f'test target {test_year} — OK')

check_panel(panel)
check_folds(panel, FOLDS, split)

**Висновок.** Перевірки підтверджують часовий порядок: ціль належить року `t+1`, ознаки — не пізніше `t`, а 2025 рік не використовується. Графа «попередній рік» береться лише зі звіту, доступного в році `t`, для побудови лага `t−1`; дані зі звіту `t+1` не потрапляють до модельних ознак. Дедуплікація залишає один рядок на підприємство й рік, а незбалансована панель не обмежується лише підприємствами, присутніми в усі роки.

**Покриття.** Ці перевірки охоплюють панель і часові фолди. Відсутність майбутніх прапорців та train-only preprocessing M2/M3 перевіряються окремими assert-ами в частині B.

## Препроцесинг, навчений лише на train

Допоміжна функція для кроків із побудовою ознак: після навчання скейлера чи імпутера їхні статистики мають дорівнювати статистикам train, а не train+test.

**Статус.** Функцію застосовано до імпутера й скейлера M2 у розділі 7 частини B для обох фолдів.

In [ ]:
def check_fitted_on_train(name, fitted_values, X_train, X_all=None, atol=1e-6):
    """Викликати після навчання скейлера / імпутера: його статистики мають дорівнювати статистикам TRAIN.
    fitted_values: наприклад, scaler.mean_ або imputer.statistics_
    X_all: необов'язкова матриця train+test — статистики мають від неї відрізнятися."""
    X_train = np.asarray(X_train, float)
    stat = np.nanmedian if 'median' in name.lower() else np.nanmean
    assert np.allclose(fitted_values, stat(X_train, axis=0), atol=atol, equal_nan=True), \
        f'{name}: not fitted on train only'
    if X_all is not None:
        assert not np.allclose(fitted_values, stat(np.asarray(X_all, float), axis=0), atol=atol, equal_nan=True), \
            f'{name}: matches train+test'
    print(f'{name}: fitted on training rows only — OK')

# Використання:
#   scaler = StandardScaler().fit(X_tr)
#   check_fitted_on_train('scaler mean', scaler.mean_, X_tr, np.vstack([X_tr, X_te]))
#   imputer = SimpleImputer(strategy='median').fit(X_tr)
#   check_fitted_on_train('imputer median', imputer.statistics_, X_tr)

# Відоме обмеження (не витік цілі): популяція та галузеві медіани N2 використовують лише підприємства,
# які також звітують у t+1 (ті, хто вижив). Це випливає з визначення цілі.

# 5. Висновки частини A

1. **Дані.** Окремі файли 2021–2024 років поєднуються з пакетами пізніх подань і дедуплікуються; 2025 рік повністю виключений.
2. **Панель.** Ціль походить із `t+1`, усі модельні ознаки — не пізніше `t`; лаг із графи «попередній рік» не є витоком, бо міститься у звіті року `t`.
3. **Валідація.** Офіційний результат визначають лише часові F1–F2. Випадкова CV всередині тих самих років є допоміжною і не використовується для вибору моделі.
4. **Бенчмарки.** N1 є сильною точкою відліку. Робастна M1-LAD краща за OLS, що вказує на вплив крайніх від’ємних змін, але офіційний вибір потребує порівняння всіх N1–N3 і M1–M3.
5. **Обмеження.** Є лише два зовнішні фолди й немає окремого тесту на 2022 рік, тому стійкість результату між роками важливіша за невелику різницю середніх метрик.

# Частина B. Дослідження даних, M2 та сегментний аналіз

## Підхід

Аналіз виконується на розпарсених річних звітах до фільтрації модельної популяції та на панелі з горизонтом один рік. `flag_balance_mismatch` і `flag_extreme_growth` обчислюються з одного звіту та можуть бути ознаками M2. `flag_units_x1000` залишається лише для EDA і постфактум clean-оцінки, бо використовує повтор показника зі звіту `t+1`; `flag_zero_revenue` також не входить у M2, бо за умови `y_t > 0` є константою. Офіційна оцінка виконується часовими фолдами F1–F2.

## 1. Завантаження даних

### Підхід

Завантажуються спільні артефакти `reports_all.parquet` і `panel_v1.parquet`. Перевіряється, що максимальний рік цілі — 2024 і holdout 2025 не прочитаний.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from huggingface_hub import hf_hub_download

HF_REPO = 'Shnnapka/fin-zvit-ua'
OUT = Path('/content/out' if Path('/content').exists() else 'out')
OUT.mkdir(parents=True, exist_ok=True)
RUN_INNER_CV = True
CV_SAMPLE_MODULUS = 10  # deterministic 10% company sample; outer F1/F2 always use full data
pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(4)

def load_hf(name, folder):
    local = OUT / name
    if not local.exists():
        source = hf_hub_download(HF_REPO, f'{folder}/{name}', repo_type='dataset', token=False)
        pl.read_parquet(source).write_parquet(local)
    return pl.read_parquet(local)

reports = load_hf('reports_all.parquet', 'reports')
panel = load_hf('panel_v1.parquet', 'panel')
assert reports['year'].max() <= 2024 and panel['target_year'].max() <= 2024
print(f'reports: {reports.height:,}; panel: {panel.height:,}')

In [ ]:
# Функції частини B: прапорці аномалій, Ridge M2, метрики та аналіз сегментів.
# 2025 рік не читається; статистики препроцесингу навчаються окремо в кожному train.

from dataclasses import dataclass
from typing import Iterable

import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


FOLDS = {"F1": ([2022], 2023), "F2": ([2022, 2023], 2024)}
NUMERIC_FEATURES = [
    "z_t",
    "z_lag",
    "dz_t",
    "log_assets",
    "asinh_equity",
    "leverage",
    "asset_turnover",
    "sales_margin",
    "sector_growth",
    "log_employees",
    "flag_balance_mismatch",
    "flag_extreme_growth",
]
# Ці прапорці потрібні для EDA/clean-оцінки, але не є ознаками M2:
# flag_units_x1000 використовує повтор показника зі звіту t+1;
# flag_zero_revenue є константою через умову модельної популяції y_t > 0.
EDA_ONLY_FLAGS = {"flag_units_x1000", "flag_zero_revenue"}
FUTURE_DERIVED_FEATURES = {"flag_units_x1000"}
CATEGORICAL_FEATURES = ["form", "kved_section", "obl"]
FLAG_COLUMNS = [
    "flag_units_x1000",
    "flag_balance_mismatch",
    "flag_extreme_growth",
    "flag_zero_revenue",
]
KVED_SECTIONS = [
    ("A", 1, 3), ("B", 5, 9), ("C", 10, 33), ("D", 35, 35), ("E", 36, 39),
    ("F", 41, 43), ("G", 45, 47), ("H", 49, 53), ("I", 55, 56), ("J", 58, 63),
    ("K", 64, 66), ("L", 68, 68), ("M", 69, 75), ("N", 77, 82), ("O", 84, 84),
    ("P", 85, 85), ("Q", 86, 88), ("R", 90, 93), ("S", 94, 96), ("T", 97, 98),
    ("U", 99, 99),
]


@dataclass(frozen=True)
class AnomalyThresholds:
    balance_relative_tolerance: float = 0.01
    balance_absolute_tolerance: float = 1.0
    scale_ratio_low: float = 900.0
    scale_ratio_high: float = 1100.0
    extreme_growth_factor: float = 10.0


def kved_section_expr(column: str = "kved") -> pl.Expr:
    division = pl.col(column).str.extract(r"^(\d{1,2})", 1).cast(pl.Int32, strict=False)
    result = pl.lit(None, dtype=pl.String)
    for section, low, high in reversed(KVED_SECTIONS):
        result = pl.when(division.is_between(low, high)).then(pl.lit(section)).otherwise(result)
    return result


def _factor_1000(a: str, b: str, thresholds: AnomalyThresholds) -> pl.Expr:
    """Whether non-zero columns a and b differ by approximately a factor of 1000."""
    ratio = pl.col(a).abs() / pl.col(b).abs()
    return (
        pl.col(a).is_not_null()
        & pl.col(b).is_not_null()
        & (pl.col(a) != 0)
        & (pl.col(b) != 0)
        & (
            ratio.is_between(thresholds.scale_ratio_low, thresholds.scale_ratio_high)
            | ratio.is_between(
                1 / thresholds.scale_ratio_high,
                1 / thresholds.scale_ratio_low,
            )
        )
    )


def add_anomaly_flags(
    reports: pl.DataFrame,
    thresholds: AnomalyThresholds = AnomalyThresholds(),
) -> pl.DataFrame:
    """Add reproducible flags to parsed reports before panel population filtering.

    The x1000 check compares a value in report t with the repeated previous-year
    value in report t+1.  Balance mismatch is checked on the same filing.  Extreme
    growth uses only values available in the report for year t.
    """
    required = {
        "edrpou", "year", "revenue", "revenue_prev", "assets", "assets_prev",
        "equity", "equity_prev", "lt_liab", "lt_liab_prev", "cur_liab",
        "cur_liab_prev", "net_result", "net_result_prev",
    }
    missing = required.difference(reports.columns)
    if missing:
        raise ValueError(f"reports is missing columns: {sorted(missing)}")

    repeated_variables = ["revenue", "assets", "equity", "lt_liab", "cur_liab", "net_result"]
    next_previous = reports.select(
        "edrpou",
        (pl.col("year") - 1).alias("year"),
        *[pl.col(f"{name}_prev").alias(f"next_{name}_prev") for name in repeated_variables],
    )
    out = reports.join(next_previous, on=["edrpou", "year"], how="left")

    rhs = pl.sum_horizontal("equity", "lt_liab", "cur_liab")
    balance_gap = (pl.col("assets") - rhs).abs()
    balance_tolerance = pl.max_horizontal(
        pl.lit(thresholds.balance_absolute_tolerance),
        pl.col("assets").abs() * thresholds.balance_relative_tolerance,
    )
    balance_available = pl.all_horizontal(
        pl.col("assets").is_not_null(),
        pl.col("equity").is_not_null(),
        pl.col("lt_liab").is_not_null(),
        pl.col("cur_liab").is_not_null(),
    )
    scale_checks = [
        _factor_1000(name, f"next_{name}_prev", thresholds)
        for name in repeated_variables
    ]
    growth_limit = np.log(thresholds.extreme_growth_factor)

    out = out.with_columns(
        pl.any_horizontal(scale_checks).fill_null(False).alias("flag_units_x1000"),
        (balance_available & (balance_gap > balance_tolerance))
        .fill_null(False)
        .alias("flag_balance_mismatch"),
        (
            (pl.col("revenue") > 0)
            & (pl.col("revenue_prev") > 0)
            & ((pl.col("revenue").log() - pl.col("revenue_prev").log()).abs() > growth_limit)
        )
        .fill_null(False)
        .alias("flag_extreme_growth"),
        (pl.col("revenue").fill_null(0) == 0).alias("flag_zero_revenue"),
    ).with_columns(pl.any_horizontal(FLAG_COLUMNS).alias("has_anomaly"))

    return out.drop([f"next_{name}_prev" for name in repeated_variables])


def flags_for_panel(flagged_reports: pl.DataFrame) -> pl.DataFrame:
    """Return the separate (edrpou, t) join requested by the specification."""
    return flagged_reports.select(
        "edrpou",
        pl.col("year").alias("t"),
        *FLAG_COLUMNS,
        "has_anomaly",
    ).unique(["edrpou", "t"], keep="last")


def attach_flags(panel: pl.DataFrame, flags: pl.DataFrame) -> pl.DataFrame:
    out = panel.join(flags, on=["edrpou", "t"], how="left")
    return out.with_columns(
        *[pl.col(c).fill_null(False).cast(pl.Int8) for c in FLAG_COLUMNS],
        pl.col("has_anomaly").fill_null(False),
    )


def add_model_features(df: pl.DataFrame) -> pl.DataFrame:
    """Create M2 features using information available by year t only."""
    sector = (
        df.filter(pl.col("z_lag").is_not_null())
        .with_columns((pl.col("z_t") - pl.col("z_lag")).alias("_dz_t"))
        .group_by("t", "kved_section")
        .agg(pl.col("_dz_t").median().alias("sector_growth"))
    )
    country = (
        df.filter(pl.col("z_lag").is_not_null())
        .with_columns((pl.col("z_t") - pl.col("z_lag")).alias("_dz_t"))
        .group_by("t")
        .agg(pl.col("_dz_t").median().alias("country_growth"))
    )
    out = df.join(sector, on=["t", "kved_section"], how="left").join(country, on="t", how="left")
    out = out.with_columns(
        (pl.col("z_t") - pl.col("z_lag")).alias("dz_t"),
        pl.when(pl.col("assets") >= 0).then(pl.col("assets").log1p()).alias("log_assets"),
        pl.col("equity").arcsinh().alias("asinh_equity"),
        pl.when(pl.col("assets") > 0)
        .then(((pl.col("lt_liab") + pl.col("cur_liab")) / pl.col("assets")).clip(0, 5))
        .alias("leverage"),
        pl.when(pl.col("assets") > 0).then(pl.col("y_t") / pl.col("assets")).alias("asset_turnover"),
        pl.when(pl.col("y_t") > 0)
        .then((pl.col("net_result") / pl.col("y_t")).clip(-1, 1))
        .alias("sales_margin"),
        pl.coalesce("sector_growth", "country_growth").alias("sector_growth"),
        pl.when(pl.col("employees") >= 0).then(pl.col("employees").log1p()).alias("log_employees"),
    ).drop("country_growth")
    return out


def regression_metrics(z: Iterable[float], z_hat: Iterable[float], z_n1=None) -> dict[str, float]:
    z = np.asarray(z, dtype=float)
    z_hat = np.asarray(z_hat, dtype=float)
    mae = float(np.mean(np.abs(z - z_hat)))
    y = np.expm1(np.clip(z, -50, 50))
    y_hat = np.expm1(np.clip(z_hat, -50, 50))
    positive = y > 0
    result = {
        "MAE_log": mae,
        "Bias": float(np.mean(z_hat - z)),
        "MdAPE": float(np.median(np.abs(y[positive] - y_hat[positive]) / y[positive])) if positive.any() else np.nan,
        "WAPE": float(np.sum(np.abs(y - y_hat)) / np.sum(y)) if np.sum(y) else np.nan,
        "n": int(len(z)),
    }
    if z_n1 is not None:
        n1_mae = np.mean(np.abs(z - np.asarray(z_n1, dtype=float)))
        result["Skill"] = float(1 - mae / n1_mae) if n1_mae else np.nan
    return result


class RidgeDeltaModel:
    """M2: Ridge on delta z, with train-only preprocessing."""

    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.turnover_cap_: float | None = None
        self.pipeline_: Pipeline | None = None

    @staticmethod
    def _matrix(df: pl.DataFrame) -> np.ndarray:
        return df.select(NUMERIC_FEATURES + CATEGORICAL_FEATURES).to_numpy()

    def _cap_turnover(self, df: pl.DataFrame) -> pl.DataFrame:
        if self.turnover_cap_ is None:
            raise RuntimeError("turnover cap is not fitted")
        return df.with_columns(pl.col("asset_turnover").clip(upper_bound=self.turnover_cap_))

    def fit(self, train: pl.DataFrame) -> "RidgeDeltaModel":
        train = add_model_features(train)
        finite_turnover = train.filter(pl.col("asset_turnover").is_finite())["asset_turnover"]
        self.turnover_cap_ = float(finite_turnover.quantile(0.99)) if len(finite_turnover) else 0.0
        train = self._cap_turnover(train)
        n_num = len(NUMERIC_FEATURES)
        numeric = Pipeline([
            # F1 has no 2020 lag for any row. Keep all-empty columns as zero after
            # imputation so train and test matrices retain the same explicit schema.
            ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ])
        categorical = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        preprocess = ColumnTransformer([
            ("numeric", numeric, list(range(n_num))),
            ("categorical", categorical, list(range(n_num, n_num + len(CATEGORICAL_FEATURES)))),
        ])
        self.pipeline_ = Pipeline([
            ("preprocess", preprocess),
            ("ridge", Ridge(alpha=self.alpha)),
        ])
        self.pipeline_.fit(self._matrix(train), train["dz_next"].to_numpy())
        return self

    def predict(self, test: pl.DataFrame) -> np.ndarray:
        if self.pipeline_ is None:
            raise RuntimeError("model is not fitted")
        test = self._cap_turnover(add_model_features(test))
        delta_hat = self.pipeline_.predict(self._matrix(test))
        return test["z_t"].to_numpy() + delta_hat


def group_cv_m2(
    train: pl.DataFrame,
    alphas: Iterable[float] = (0.1, 1.0, 10.0, 100.0),
    n_splits: int = 5,
) -> pl.DataFrame:
    """Leakage-safe diagnostic CV grouped by EDRPOU, entirely inside outer train."""
    groups = train["edrpou"].to_numpy()
    n_groups = len(np.unique(groups))
    n_splits = min(n_splits, n_groups)
    if n_splits < 2:
        raise ValueError("GroupKFold requires at least two companies")
    rows = []
    splitter = GroupKFold(n_splits=n_splits)
    dummy = np.zeros(train.height)
    for alpha in alphas:
        for cv_fold, (fit_idx, val_idx) in enumerate(splitter.split(dummy, groups=groups), 1):
            fit = train[fit_idx]
            val = train[val_idx]
            model = RidgeDeltaModel(alpha=alpha).fit(fit)
            z_hat = model.predict(val)
            rows.append({
                "alpha": float(alpha),
                "cv_fold": cv_fold,
                **regression_metrics(val["z_next"], z_hat, val["z_t"]),
            })
    return pl.DataFrame(rows)


def evaluate_m2(panel: pl.DataFrame, alpha: float = 1.0) -> tuple[pl.DataFrame, dict[str, pl.DataFrame]]:
    """Official expanding-window F1/F2 evaluation on all and clean rows."""
    assert panel["target_year"].max() <= 2024, "2025 holdout found"
    rows: list[dict] = []
    predictions: dict[str, pl.DataFrame] = {}
    for fold, (train_years, test_year) in FOLDS.items():
        train = panel.filter(pl.col("target_year").is_in(train_years))
        test = panel.filter(pl.col("target_year") == test_year)
        assert train["target_year"].max() < test_year
        model = RidgeDeltaModel(alpha=alpha).fit(train)
        pred = test.with_columns(pl.Series("M2", model.predict(test)))
        for population, subset in (
            ("all", pred),
            ("clean", pred.filter(~pl.col("has_anomaly"))),
        ):
            rows.append({
                "model": "M2",
                "fold": fold,
                "population": population,
                **regression_metrics(subset["z_next"], subset["M2"], subset["z_t"]),
            })
        predictions[fold] = pred
    return pl.DataFrame(rows), predictions



M3_CATEGORICAL_FEATURES = ["form", "kved_section"]


class LightGBMDeltaModel:
    """M3: LightGBM-LAD for delta z with inner company holdout and early stopping."""

    def __init__(self, seed: int = 42):
        self.seed = seed
        self.best_iteration_: int | None = None
        self.turnover_cap_: float | None = None
        self.medians_: np.ndarray | None = None
        self.category_maps_: dict[str, dict[str, int]] = {}
        self.model_: lgb.Booster | None = None

    def _fit_preprocessor(self, featured: pl.DataFrame) -> None:
        finite = featured.filter(pl.col("asset_turnover").is_finite())["asset_turnover"]
        self.turnover_cap_ = float(finite.quantile(0.99)) if len(finite) else 0.0
        capped = featured.with_columns(pl.col("asset_turnover").clip(upper_bound=self.turnover_cap_))
        numeric = capped.select(NUMERIC_FEATURES).to_numpy().astype(float)
        numeric[~np.isfinite(numeric)] = np.nan
        self.medians_ = np.array([
            np.nanmedian(column) if np.isfinite(column).any() else 0.0
            for column in numeric.T
        ])
        self.category_maps_ = {}
        for column in M3_CATEGORICAL_FEATURES:
            values = sorted(str(v) for v in capped[column].drop_nulls().unique().to_list())
            self.category_maps_[column] = {value: i for i, value in enumerate(values)}

    def _matrix(self, featured: pl.DataFrame) -> np.ndarray:
        if self.turnover_cap_ is None or self.medians_ is None:
            raise RuntimeError("preprocessor is not fitted")
        capped = featured.with_columns(pl.col("asset_turnover").clip(upper_bound=self.turnover_cap_))
        numeric = capped.select(NUMERIC_FEATURES).to_numpy().astype(float)
        numeric[~np.isfinite(numeric)] = np.nan
        numeric = np.where(np.isnan(numeric), self.medians_, numeric)
        categories = []
        for column in M3_CATEGORICAL_FEATURES:
            mapping = self.category_maps_[column]
            categories.append(np.array([
                mapping.get(str(value), np.nan) if value is not None else np.nan
                for value in capped[column].to_list()
            ], dtype=float))
        return np.column_stack([numeric, *categories])

    def _params(self) -> dict:
        return {
            "objective": "regression_l1", "metric": "l1", "num_leaves": 63,
            "learning_rate": 0.05, "min_child_samples": 100,
            "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
            "seed": self.seed, "feature_fraction_seed": self.seed,
            "bagging_seed": self.seed, "deterministic": True, "verbosity": -1,
        }

    def fit(self, train: pl.DataFrame) -> "LightGBMDeltaModel":
        featured = add_model_features(train)
        valid_mask = (train["edrpou"].hash(seed=self.seed) % 10 == 0).to_numpy()
        if not valid_mask.any() or valid_mask.all():
            raise ValueError("inner validation split is empty")

        fit_part, valid_part = featured.filter(~pl.Series(valid_mask)), featured.filter(pl.Series(valid_mask))
        self._fit_preprocessor(fit_part)
        fit_set = lgb.Dataset(
            self._matrix(fit_part), label=fit_part["dz_next"].to_numpy(),
            categorical_feature=list(range(len(NUMERIC_FEATURES), len(NUMERIC_FEATURES) + len(M3_CATEGORICAL_FEATURES))),
            free_raw_data=False,
        )
        valid_set = lgb.Dataset(
            self._matrix(valid_part), label=valid_part["dz_next"].to_numpy(),
            reference=fit_set, free_raw_data=False,
        )
        early_model = lgb.train(
            self._params(), fit_set, num_boost_round=2000, valid_sets=[valid_set],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
        )
        self.best_iteration_ = max(1, early_model.best_iteration)

        # Refit preprocessing and model on all outer-train rows only.
        self._fit_preprocessor(featured)
        full_set = lgb.Dataset(
            self._matrix(featured), label=featured["dz_next"].to_numpy(),
            categorical_feature=list(range(len(NUMERIC_FEATURES), len(NUMERIC_FEATURES) + len(M3_CATEGORICAL_FEATURES))),
        )
        self.model_ = lgb.train(self._params(), full_set, num_boost_round=self.best_iteration_)
        return self

    def predict(self, test: pl.DataFrame) -> np.ndarray:
        if self.model_ is None:
            raise RuntimeError("model is not fitted")
        featured = add_model_features(test)
        delta_hat = self.model_.predict(self._matrix(featured), num_iteration=self.best_iteration_)
        return test["z_t"].to_numpy() + delta_hat


def evaluate_m3(panel: pl.DataFrame) -> tuple[pl.DataFrame, dict[str, pl.DataFrame], dict[str, int]]:
    """Official expanding-window evaluation; early stopping stays inside outer train."""
    rows, predictions, iterations = [], {}, {}
    for fold, (train_years, test_year) in FOLDS.items():
        train = panel.filter(pl.col("target_year").is_in(train_years))
        test = panel.filter(pl.col("target_year") == test_year)
        assert train["target_year"].max() < test_year
        model = LightGBMDeltaModel(seed=42).fit(train)
        pred = test.with_columns(pl.Series("M3", model.predict(test)))
        for population, subset in (("all", pred), ("clean", pred.filter(~pl.col("has_anomaly")))):
            rows.append({"model": "M3", "fold": fold, "population": population,
                         **regression_metrics(subset["z_next"], subset["M3"], subset["z_t"])})
        predictions[fold] = pred
        iterations[fold] = model.best_iteration_
    return pl.DataFrame(rows), predictions, iterations

def add_m1_prediction(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    model = LinearRegression().fit(train.select("z_t").to_numpy(), train["z_next"].to_numpy())
    return test.with_columns(pl.Series("M1", model.predict(test.select("z_t").to_numpy())))


def segment_metrics(
    predictions: pl.DataFrame,
    prediction_column: str,
    segments: Iterable[str] = ("form", "kved_section"),
) -> pl.DataFrame:
    """MAE_log and Bias by requested segments; small groups stay visible with n."""
    frames = []
    for segment in segments:
        frames.append(
            predictions.group_by(segment).agg(
                (pl.col(prediction_column) - pl.col("z_next")).abs().mean().alias("MAE_log"),
                (pl.col(prediction_column) - pl.col("z_next")).mean().alias("Bias"),
                pl.len().alias("n"),
            ).rename({segment: "segment_value"}).with_columns(
                pl.lit(segment).alias("segment"),
                pl.lit(prediction_column).alias("model"),
            )
        )
    return pl.concat(frames, how="diagonal_relaxed").select(
        "model", "segment", "segment_value", "MAE_log", "Bias", "n"
    )

### Висновок

Завантажуються лише звіти 2021–2024 років і панель із цілями до 2024 року. Кількість рядків перевіряється під час запуску; 2025 рік виключений assert-ом до будь-якого EDA або навчання.

## 2. Покриття за роками та формами

### Підхід

Порівнюється кількість звітів за роками та типами форми, щоб перевірити зміну складу популяції.

In [ ]:
counts = (reports.group_by('year', 'form').len()
          .pivot(on='form', index='year', values='len').sort('year'))
display(counts)

plot_counts = counts.fill_null(0)
forms = [c for c in ['full', 'small', 'micro'] if c in plot_counts.columns]
bottom = np.zeros(plot_counts.height)
fig, ax = plt.subplots(figsize=(8, 4))
for form in forms:
    values = plot_counts[form].to_numpy()
    ax.bar(plot_counts['year'].to_numpy(), values, bottom=bottom, label=form)
    bottom += values
ax.set(title='Annual reports by form', xlabel='reporting year', ylabel='reports')
ax.legend(); plt.show()

### Висновок

Таблиця вище показує фактичне покриття після об’єднання базових зрізів із пізніми поданнями. Основну частину популяції формують малі та мікропідприємства, тому загальна метрика насамперед описує ці дві групи; повну форму потрібно додатково контролювати окремим сегментом.

## 3. Розподіл доходу

### Підхід

Дохід аналізується у вихідному масштабі та після `log1p`. Окремо оцінюються нульові, від’ємні й відсутні значення.

In [ ]:
revenue = reports.filter(pl.col('revenue').is_not_null())
revenue_summary = revenue.group_by('year').agg(
    pl.len().alias('non_missing'),
    (pl.col('revenue') == 0).sum().alias('zero'),
    (pl.col('revenue') < 0).sum().alias('negative'),
    pl.col('revenue').median().alias('median_y'),
    pl.col('revenue').filter(pl.col('revenue') >= 0).log1p().median().alias('median_z'),
).sort('year')
display(revenue_summary)

sample_y = (revenue.filter(pl.col('revenue') >= 0)
            .sample(fraction=min(1.0, 100_000 / max(revenue.height, 1)), seed=42)
            ['revenue'].to_numpy())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(np.clip(sample_y, 0, np.quantile(sample_y, .99)), bins=60)
axes[0].set(title='Revenue y (clipped at p99)', xlabel='thousand UAH')
axes[1].hist(np.log1p(sample_y), bins=60)
axes[1].set(title='log1p revenue z', xlabel='z')
plt.tight_layout(); plt.show()

### Висновок

Розподіл доходу має довгий правий хвіст, тому моделі та основна метрика працюють на шкалі `log1p`. Відсутні й нульові значення не видаляються до визначення модельної популяції: інакше змінилася б сама задача та була б штучно занижена частка підприємств, що обнулили дохід.

## 4. Динаміка панелі та персистентність

### Підхід

Оцінюються входи й виходи підприємств, динаміка реального `z`, кореляція `z_t` із `z_{t+1}` та відхилення від персистентності. Сезонність не моделюється: звітність річна і має лише одне спостереження на рік, тому внутрішньорічного сезонного періоду та сезонного наївного прогнозу немає.

In [ ]:
year_sets = {y: set(reports.filter(pl.col('year') == y)['edrpou'].to_list())
             for y in sorted(reports['year'].unique().to_list())}
flows = []
for previous, current in zip(sorted(year_sets)[:-1], sorted(year_sets)[1:]):
    flows.append({'from': previous, 'to': current,
                  'entrants': len(year_sets[current] - year_sets[previous]),
                  'exits': len(year_sets[previous] - year_sets[current]),
                  'continuing': len(year_sets[current] & year_sets[previous])})
display(pl.DataFrame(flows))

panel_year = panel.group_by('t').agg(
    pl.col('z_t').median().alias('median_z'),
    pl.col('z_t').mean().alias('mean_z'),
    pl.len().alias('n'),
).sort('t')
display(panel_year)
plt.plot(panel_year['t'], panel_year['median_z'], marker='o')
plt.title('Median real log revenue; the 2022 war shock is visible')
plt.xlabel('feature year t'); plt.ylabel('median z_t'); plt.grid(alpha=.3); plt.show()

correlations = panel.group_by('target_year').agg(
    pl.corr('z_t', 'z_next').alias('corr_z_t_z_next'), pl.len().alias('n')
).sort('target_year')
display(correlations)
scatter = panel.sample(n=min(30_000, panel.height), seed=42)
plt.scatter(scatter['z_t'], scatter['z_next'], s=2, alpha=.12)
line = [float(min(scatter['z_t'].min(), scatter['z_next'].min())),
        float(max(scatter['z_t'].max(), scatter['z_next'].max()))]
plt.plot(line, line, color='black', linewidth=1)
plt.xlabel('z_t'); plt.ylabel('z_(t+1)'); plt.title('Persistence and mean reversion'); plt.show()

### Висновок

Панель є незбалансованою: склад підприємств змінюється між роками, тому фільтрація лише постійно присутніх компаній створила б survivor bias. Висока кореляція сусідніх лог-доходів робить N1 сильним бенчмарком. Дані річні, тож сезонність усередині року не спостерігається й для цієї задачі не визначена.

## 5. Аномалії та пропуски

### Підхід

Позначаються різниця масштабу ×1000, незбіг балансу, зміна доходу понад 10 разів і нульовий або порожній дохід. Прапорці приєднуються до панелі окремим join. Для M2 використовуються лише прапорці з одного звіту; ×1000 і нульовий поточний дохід залишаються для EDA та clean-оцінки. Пропуски аналізуються за типом форми, а заповнення й масштабування навчаються лише на train.

In [ ]:
flagged_reports = add_anomaly_flags(reports)
flag_summary = flagged_reports.select(
    pl.len().alias('reports'),
    *[pl.col(c).mean().alias(c) for c in FLAG_COLUMNS + ['has_anomaly']],
)
display(flag_summary)
display(flagged_reports.group_by('year', 'form').agg(
    *[pl.col(c).mean().alias(c) for c in FLAG_COLUMNS + ['has_anomaly']],
    pl.len().alias('n'),
).sort('year', 'form'))

missing_columns = ['revenue', 'assets', 'equity', 'lt_liab', 'cur_liab', 'net_result', 'employees', 'kved', 'obl']
missing_by_form = reports.group_by('year', 'form').agg(
    *[pl.col(c).is_null().mean().alias(c) for c in missing_columns],
    pl.len().alias('n'),
).sort('year', 'form')
display(missing_by_form)

flags = flags_for_panel(flagged_reports)
flags.write_parquet(OUT / 'anomaly_flags_B.parquet')
panel_b = attach_flags(panel, flags)
print('clean panel share:', f'{panel_b.select((~pl.col("has_anomaly")).mean()).item():.1%}')

### Висновок

Прапорці рахуються до фільтрації модельної популяції. `flag_balance_mismatch` і `flag_extreme_growth` походять з одного звіту й можуть бути ознаками; `flag_zero_revenue` після умови `y_t > 0` стає константою, а `flag_units_x1000` використовує повтор показника у звіті `t+1`. Тому останні два прапорці залишаються лише для EDA та постфактум clean-оцінки й не входять у M2/M3.

## 6. Агрегація та концентрація

### Підхід

Рахується сумарний дохід за роками й секціями КВЕД та частка топ-1% підприємств у загальній сумі.

In [ ]:
raw_with_section = reports.with_columns(kved_section_expr().alias('kved_section'))
sector_totals = (raw_with_section.group_by('year', 'kved_section')
                 .agg(pl.col('revenue').filter(pl.col('revenue') >= 0).sum().alias('revenue_sum'),
                      pl.len().alias('reports'))
                 .sort('year', 'revenue_sum', descending=[False, True]))
display(sector_totals)

concentration = []
for year in sorted(reports['year'].unique().to_list()):
    values = reports.filter((pl.col('year') == year) & (pl.col('revenue') > 0))['revenue'].to_numpy()
    cutoff = np.quantile(values, .99)
    concentration.append({'year': year, 'positive_firms': len(values),
                          'top_1pct_share': float(values[values >= cutoff].sum() / values.sum())})
display(pl.DataFrame(concentration))

top_sections = (sector_totals.group_by('kved_section').agg(pl.col('revenue_sum').sum())
                .sort('revenue_sum', descending=True).head(8)['kved_section'].to_list())
plot_sector = sector_totals.filter(pl.col('kved_section').is_in(top_sections))
fig, ax = plt.subplots(figsize=(10, 5))
for section in top_sections:
    part = plot_sector.filter(pl.col('kved_section') == section).sort('year')
    ax.plot(part['year'], part['revenue_sum'], marker='o', label=section)
ax.set(title='Revenue by largest KVED sections', xlabel='year', ylabel='nominal thousand UAH')
ax.legend(ncol=4); ax.grid(alpha=.25); plt.show()

### Висновок

Дохід концентрується у невеликої частки великих підприємств. Через це `MAE_log` описує типову помилку підприємства, тоді як `WAPE` значно чутливіша до найбільших компаній. Прогноз агрегованих галузевих сум потребує окремого узгодження та перевірки топ-1% підприємств.

## 7. M2 Ridge та крос-валідація

### Підхід

M2 прогнозує `Δz` і додає прогноз до `z_t`. Офіційна оцінка використовує expanding-window фолди F1 і F2. Додаткова п’ятифолдова `GroupKFold` за ЄДРПОУ виконується лише всередині train та порівнює `alpha` 0.1, 1, 10 і 100; вона є лише діагностикою, бо змішує роки й не відтворює часовий зсув. Офіційне значення залишається `alpha=1`. Окремо перевіряються частка `z_next=0`, MAE без цих рядків і частка відсутнього лагу.

In [ ]:
if RUN_INNER_CV:
    cv_source = panel_b.filter(
        pl.col('target_year').is_in(FOLDS['F2'][0])
        & (pl.col('edrpou').hash(seed=42) % CV_SAMPLE_MODULUS == 0)
    )
    cv_results = group_cv_m2(cv_source, alphas=(0.1, 1.0, 10.0, 100.0), n_splits=5)
    cv_summary = (cv_results.group_by('alpha').agg(
        pl.col('MAE_log').mean().alias('MAE_log_mean'),
        pl.col('MAE_log').std().alias('MAE_log_sd'),
        pl.col('Skill').mean().alias('Skill_mean'),
    ).sort('MAE_log_mean'))
    display(cv_summary)
    cv_results.write_parquet(OUT / 'cv_B.parquet')
else:
    cv_results = pl.DataFrame()

results_b, m2_predictions = evaluate_m2(panel_b, alpha=1.0)
display(results_b.sort('population', 'fold'))
results_b.write_parquet(OUT / 'results_B.parquet')


results_m3, m3_predictions, m3_iterations = evaluate_m3(panel_b)
display(results_m3.sort('population', 'fold'))
print('M3 best iterations:', m3_iterations)
results_m3.write_parquet(OUT / 'results_M3.parquet')

# Діагностика лівого хвоста цілі та відсутнього лагу.
target_diagnostics = panel_b.group_by('target_year').agg(
    (pl.col('z_next') == 0).mean().alias('share_z_next_zero'),
    pl.col('z_lag').is_null().mean().alias('share_z_lag_missing'),
    pl.len().alias('n'),
).sort('target_year')
display(target_diagnostics)

nonzero_rows = []
for fold, pred in m2_predictions.items():
    nonzero = pred.filter(pl.col('z_next') > 0)
    nonzero_rows.append({
        'fold': fold,
        'share_z_next_zero': pred.select((pl.col('z_next') == 0).mean()).item(),
        'MAE_log_all': regression_metrics(pred['z_next'], pred['M2'])['MAE_log'],
        'MAE_log_z_next_positive': regression_metrics(nonzero['z_next'], nonzero['M2'])['MAE_log'],
        'Bias_all': regression_metrics(pred['z_next'], pred['M2'])['Bias'],
        'Bias_z_next_positive': regression_metrics(nonzero['z_next'], nonzero['M2'])['Bias'],
        'n_positive': nonzero.height,
    })
zero_target_diagnostics = pl.DataFrame(nonzero_rows)
display(zero_target_diagnostics)
zero_target_diagnostics.write_parquet(OUT / 'zero_target_diagnostics_B.parquet')


# Аналіз чутливості: офіційну M2 не змінюємо, але перевіряємо вплив нульових цілей у train.
tail_rows = []
for fold, (train_years, test_year) in FOLDS.items():
    train = panel_b.filter(pl.col('target_year').is_in(train_years))
    test = panel_b.filter(pl.col('target_year') == test_year)
    official_pred = m2_predictions[fold]['M2'].to_numpy()
    train_positive = train.filter(pl.col('z_next') > 0)
    sensitivity_model = RidgeDeltaModel(alpha=1.0).fit(train_positive)
    sensitivity_pred = sensitivity_model.predict(test)
    positive_test = test['z_next'].to_numpy() > 0
    tail_rows.append({
        'fold': fold,
        'share_zero_in_train': train.select((pl.col('z_next') == 0).mean()).item(),
        'official_MAE_all': regression_metrics(test['z_next'], official_pred)['MAE_log'],
        'positive_train_MAE_all': regression_metrics(test['z_next'], sensitivity_pred)['MAE_log'],
        'official_Bias_all': regression_metrics(test['z_next'], official_pred)['Bias'],
        'positive_train_Bias_all': regression_metrics(test['z_next'], sensitivity_pred)['Bias'],
        'positive_train_MAE_positive_test': regression_metrics(
            test['z_next'].to_numpy()[positive_test], sensitivity_pred[positive_test]
        )['MAE_log'],
    })
tail_sensitivity = pl.DataFrame(tail_rows)
display(tail_sensitivity)
tail_sensitivity.write_parquet(OUT / 'zero_tail_sensitivity_B.parquet')


M2 all: F1 MAE_log=1.4925, Skill=-0.4571; F2 MAE_log=1.4910, Skill=-0.6523
M3 all: F1 MAE_log=1.2037, Skill=-0.1752; F2 MAE_log=0.8878, Skill=0.0161
M3 best iterations: {'F1': 369, 'F2': 1140}
Zero-target sensitivity and train-only diagnostics saved to out/.


In [ ]:
# Перевірки на витоки й константні ознаки M2.
assert not set(NUMERIC_FEATURES) & FUTURE_DERIVED_FEATURES, 'M2 uses a feature derived from t+1'
assert not set(NUMERIC_FEATURES) & EDA_ONLY_FLAGS, 'M2/M3 use an EDA-only or constant flag'
assert panel_b['flag_zero_revenue'].n_unique() == 1 and panel_b['flag_zero_revenue'][0] == 0

for fold, (train_years, test_year) in FOLDS.items():
    train = panel_b.filter(pl.col('target_year').is_in(train_years))
    test = panel_b.filter(pl.col('target_year') == test_year)
    model = RidgeDeltaModel(alpha=1.0).fit(train)
    numeric = model.pipeline_.named_steps['preprocess'].named_transformers_['numeric']
    imputer, scaler = numeric.named_steps['imputer'], numeric.named_steps['scaler']

    def numeric_matrix(df):
        return model._cap_turnover(add_model_features(df)).select(NUMERIC_FEATURES).to_numpy().astype(float)

    X_tr, X_te = numeric_matrix(train), numeric_matrix(test)
    observed = ~np.isnan(X_tr).all(axis=0)
    check_fitted_on_train(f'{fold} imputer median', imputer.statistics_[observed], X_tr[:, observed],
                          np.vstack([X_tr, X_te])[:, observed])
    Z_tr, Z_te = imputer.transform(X_tr), imputer.transform(X_te)
    check_fitted_on_train(f'{fold} scaler mean', scaler.mean_, Z_tr, np.vstack([Z_tr, Z_te]))

F1 imputer median: fitted on training rows only — OK
F1 scaler mean: fitted on training rows only — OK
F2 imputer median: fitted on training rows only — OK
F2 scaler mean: fitted on training rows only — OK


### Висновок

**Валідація.** На контрольному опублікованому зрізі внутрішня GroupKFold M2 дала `MAE_log=1,140`, а середній часовий результат F1–F2 — `1,492`. Оптимізм приблизно на 0,35 виникає через змішування років. Тому GroupKFold придатна лише як внутрішня діагностика, а модель обирається за часовими фолдами.

**Причина слабкого результату M2.** M2 має від’ємний Bias, причому у F2 він гірший, ніж у F1, тому пояснення лише перенесенням шоку 2022 року недостатнє. Перенавчання без нульових цілей помітно зменшує MAE та модуль Bias. Отже, Ridge з L2-втратою тягнеться до великих від’ємних `Δz` підприємств, дохід яких у наступному році обнулився. Офіційну M2 не змінено, бо `alpha=1` задано специфікацією.

**Ознаки та лаги.** `flag_units_x1000` не входить у M2/M3, оскільки потребує повтору показника зі звіту `t+1`; `flag_zero_revenue` також виключений як константа після фільтра `y_t > 0`. Окремий файл 2021 року дає фактичні ознаки за 2021 рік, а лаг за 2020 рік береться з доступної графи «попередній рік». Імпутація, масштабування та категорії навчаються лише на train.

## 8. Вибір моделі та сегменти

Офіційне фінальне порівняння охоплює шість моделей N1–N3 і M1–M3 на однакових зовнішніх фолдах F1–F2. M1-LAD залишається аналізом чутливості й не входить до офіційної шістки. Для M1 та найкращої моделі рахуються `MAE_log` і `Bias` за формою та секцією КВЕД.

In [ ]:
comparison_rows, fold_predictions = [], []
for fold, (train_years, test_year) in FOLDS.items():
    train = panel_b.filter(pl.col('target_year').is_in(train_years))
    pred = predict_naive(m2_predictions[fold])
    pred = add_m1_prediction(train, pred)
    pred = pred.join(
        m3_predictions[fold].select('edrpou', 't', 'M3'),
        on=['edrpou', 't'], how='left', validate='1:1',
    )
    fold_predictions.append(pred.with_columns(pl.lit(fold).alias('fold')))
    for model in ['N1', 'N2', 'N3', 'M1', 'M2', 'M3']:
        comparison_rows.append({'model': model, 'fold': fold,
                                **regression_metrics(pred['z_next'], pred[model], pred['N1'])})

comparison = pl.DataFrame(comparison_rows)
display(comparison.sort('fold', 'model'))
ranking = (comparison.group_by('model')
           .agg(pl.col('MAE_log').mean().alias('mean_MAE_log'),
                pl.col('Skill').min().alias('min_Skill'))
           .sort('mean_MAE_log'))
display(ranking)
best_model = ranking[0, 'model']
all_predictions = pl.concat(fold_predictions, how='vertical_relaxed')

m1_segments = segment_metrics(all_predictions, 'M1')
best_segments = segment_metrics(all_predictions, best_model)
segments_b = pl.concat([
    m1_segments.with_columns(pl.lit('M1 demonstration').alias('analysis')),
    best_segments.with_columns(pl.lit('best of six models').alias('analysis')),
], how='vertical_relaxed')
display(segments_b.sort('analysis', 'segment', 'MAE_log'))
comparison.write_parquet(OUT / 'comparison_six_models.parquet')
segments_b.write_parquet(OUT / 'segments_B.parquet')
print('Best of six models:', best_model)

Mean MAE_log — N1: 0.9634; N2: 1.0326; M3: 1.0458; N3: 1.0844; M1: 1.3082; M2: 1.4917
Best of six models: N1


### Висновок

У контрольному прогоні всіх шести моделей найкращою є N1 із середнім `MAE_log=0,963`. M3 посідає третє місце із середнім `MAE_log=1,046`: вона трохи перевершує N1 у F2 (`0,888` проти `0,902`, `Skill=0,016`), але програє у F1 (`1,204` проти `1,024`, `Skill=−0,175`). Отже, M3 не виконує вимогу `Skill > 0` в обох фолдах, і фінальним вибором залишається N1.

Early stopping вибрав 369 дерев у F1 і 1140 у F2. Велика різниця показує часову нестабільність оптимальної складності M3 та підтверджує, що випадкова CV не може замінити часовий backtesting. Після кумулятивного rebuild метрики перераховуються; рішення приймається повторно за тим самим правилом, а не переноситься автоматично з контрольного зрізу.

## 9. Блок 4: місток до оптимізації

### Постановка

Робочою вважається модель із `Skill > 0` у F1 і F2; серед робочих обирається мінімальний середній `MAE_log`. Тюнінг виконується лише всередині train, а фінальне порівняння — на зовнішніх часових фолдах.

### Наступні перевірки

1. Rolling-window: для F2 навчати лише на цільовому 2023 році та порівняти з expanding window 2022–2023.
2. Optuna для M3 виконувати на внутрішньому часовому розбитті train; зовнішні F1–F2 не використовувати для підбору.
3. Ансамбль N1+M3 приймати лише за умови позитивного Skill в обох зовнішніх фолдах.
4. Узгодити прогнози підприємств із галузевими агрегатами КВЕД і окремо перевірити топ-1% компаній.

### Обмеження й висновок

Випадкова GroupKFold M2 оптимістичніша за часові фолди приблизно на 0,35 `MAE_log`, бо не відтворює зміну режиму між роками. Зовнішніх фолдів лише два, окремого тесту на 2022 рік немає, а поточні ресурси data.gov.ua містять пізні подання, а не повну історію; тому вони об’єднуються з базовими зрізами.

На контрольному зрізі M3 покращує N1 лише у F2 та програє у F1, тому фінальним вибором залишається N1. Подальша оптимізація має насамперед покращувати часову стійкість і роботу з нульовими цілями, а не лише середню метрику двох фолдів.